<a href="https://colab.research.google.com/github/kashafejaz50-cell/Flyrank_ML_Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kashafejaz50-cell/Flyrank_ML_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!pip -q install duckdb datasets huggingface_hub pyarrow

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN:
    print("✅ Hugging Face token loaded successfully!")
else:
    print("❌ Token not found!")

✅ Hugging Face token loaded successfully!


In [ ]:
from huggingface_hub import login

login(token=HF_TOKEN)

In [ ]:
from datasets import get_dataset_config_names

configs = get_dataset_config_names(
    "FlyRank/internship-warehouse",
    token=HF_TOKEN
)

print(configs)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

['dim_clients', 'dim_content', 'fact_content_daily_performance', 'fact_content_query_90d']


# Setup
Import libraries and connect to the FlyRank warehouse.

In [ ]:
# Install required packages (run once per Colab session)
!pip -q install duckdb huggingface_hub pyarrow

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN:
    print("✅ HF_TOKEN loaded successfully.")
else:
    raise ValueError("HF_TOKEN not found in Colab Secrets.")

✅ HF_TOKEN loaded successfully.


In [ ]:
#Connect DuckDB to HuggingFace
import duckdb

con = duckdb.connect()

# Register your Hugging Face token as a DuckDB secret
con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("✅ DuckDB connected to Hugging Face.")

✅ DuckDB connected to Hugging Face.


In [ ]:
march_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

query = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{march_path}')
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [ ]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{march_path}')
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [ ]:
#Grain Verification Query
grain_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_rows
FROM read_parquet('{march_path}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
"""

con.sql(grain_query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
One row represents the daily performance of one content item for one client on one reporting date.

For this assignment, I use the `fact_content_daily_performance` table and analyze data from March 2026 (2026-03-01 to 2026-03-31). I verify the unit of analysis and time window using the queries below.

In [ ]:
#data range
query = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{march_path}')
"""

con.sql(query).df()

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [ ]:
#grain verification
grain_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_rows
FROM read_parquet('{march_path}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
"""

con.sql(grain_query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_rows


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions

These fields describe search visibility and user engagement. They are available before making the content refresh decision, so they are safe to use as input features.

### Label / Proxy
- Content refresh priority (ranking score)

The objective is to rank content pages according to their priority for refresh. The final ranking should be based on future observed performance rather than information available at the decision time.

### Context
- report_date
- client_hash_id
- content_hash_id
- month

These fields identify each record, define the time period, and support filtering or grouping. They are not used as model features.

### Excluded
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available

These fields indicate data availability and system configuration rather than content performance. They are useful for validating the dataset and filtering records but are excluded from model features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Display the fields referenced in the data contract

fields = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    month,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions,
    client_has_gsc,
    client_has_ga4,
    gsc_data_available,
    ga4_data_available
FROM read_parquet('{march_path}')
LIMIT 5
""").df()

fields


,report_date,client_hash_id,content_hash_id,month,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03,20,0,3.350000,<NA>,<NA>,True,False,True,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03,1,0,0.000000,<NA>,<NA>,True,False,True,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03,125,1,4.928000,<NA>,<NA>,True,False,True,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03,7,0,4.000000,<NA>,<NA>,True,False,True,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03,11,0,2.272727,<NA>,<NA>,True,False,True,<NA>


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I verified the main claims of my data contract using SQL queries on the March 2026 partition of the warehouse.

- **Grain:** Verified that one row represents one content item for one client on one reporting date by checking for duplicate combinations.
- **Counts:** Verified the total number of rows in the selected month.
- **Time Window:** Confirmed the date range of the selected partition.
- **Missing Values / Availability:** Checked how many records have Google Search Console and Google Analytics data available before using them as features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Grain verification
grain_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_rows
FROM read_parquet('{march_path}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
"""

con.sql(grain_query).df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_rows


In [ ]:
#Counts + Time Window
count_query = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{march_path}')
"""

con.sql(count_query).df()

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [ ]:
#Data Availability (Missingness Check)
availability_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available
FROM read_parquet('{march_path}')
"""

con.sql(availability_query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available,ga4_available
0,9841378,3611061,413966


In [ ]:
#Missing Value Check
missing_query = f"""
SELECT
    AVG(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS missing_avg_position,
    AVG(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) AS missing_sessions
FROM read_parquet('{march_path}')
"""

con.sql(missing_query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,missing_avg_position,missing_sessions
0,0.633074,0.30674


## 4. Data limits

The warehouse data has several limitations that should be considered when interpreting results.

- Client history is not balanced. Some clients have much longer data histories than others, so comparisons across clients may not always be fair.
- Google Analytics (GA4) data is unavailable before a client's GA4 integration. Rows from that period contain unavailable analytics data, so only records with `ga4_data_available IS TRUE` should be used for GA4-based analysis.
- Google Search Console (GSC) availability also varies across clients, so analyses should use only records where `gsc_data_available IS TRUE`.
- Time windows can overlap if features and labels are not defined carefully. Features should always come from information available before the prediction or ranking decision to avoid data leakage.
- This dataset supports decision-making for content refresh but cannot explain external reasons for performance changes, such as algorithm updates, seasonal trends, or marketing campaigns.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Verify data availability across the selected month

limits_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available,
    COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS gsc_unavailable,
    COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS ga4_unavailable
FROM read_parquet('{march_path}')
"""

con.sql(limits_query).df()


,total_rows,gsc_available,ga4_available,gsc_unavailable,ga4_unavailable
0,9841378,3611061,413966,6230317,6408671


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.